In [9]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('titanic.csv')

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
# Feature 2: IsAlone (1 if traveling alone, 0 otherwise)
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

X = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin', 'Survived'], axis=1)
y = df['Survived']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numerical_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize', 'IsAlone']
categorical_cols = ['Pclass', 'Sex', 'Embarked']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, solver='liblinear'))
])

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Pipeline Accuracy with Feature Engineering: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

# Save the Final Pipeline using joblib (creates titanic_pipeline.pkl)
joblib.dump(pipeline, 'titanic_pipeline.pkl')
print("\nPipeline successfully saved as 'titanic_pipeline.pkl'!")

Pipeline Accuracy with Feature Engineering: 79.33%

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.86      0.83       105
           1       0.78      0.70      0.74        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179


Pipeline successfully saved as 'titanic_pipeline.pkl'!


**Pipeline & Feature Engineering Summary:**

**Feature Engineering:** We created two new features—FamilySize (total family members aboard) and IsAlone (a binary flag indicating if a passenger traveled by themselves). These successfully captured additional behavioral patterns that improved predictive quality.

**Why Pipelines Matter:** By wrapping our imputers, scalers, encoders, and classifiers into a single Pipeline object, we completely eliminate data leakage. The transformers learn scaling and encoding parameters strictly on the training folds during cross-validation or fitting, rather than bleeding information from the test set. It also ensures that any future data sent to production goes through the exact same automated transformation steps cleanly.